In [1]:
import cv2
import numpy as np
import os

from skimage.feature import hog

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)


# ============================================================
# SETTINGS
# ============================================================

DATA_FOLDER = r"C:\Users\PC-LAB1\Desktop\shok\pictures\train"

IMAGES_PER_CLASS = 4000

N_SPLITS = 5

PCA_VARIANCE = 0.95


# ============================================================
# FEATURE EXTRACTION
# ============================================================

def extract_features(image_path):

    img = cv2.imread(image_path)

    if img is None:
        return None

    img = cv2.resize(img, (64, 64))


    # ========================================================
    # 1. HSV COLOR HISTOGRAM
    # 96 FEATURES
    # ========================================================

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    hist_features = []

    for channel in range(3):

        if channel == 0:
            hist_range = [0, 180]
        else:
            hist_range = [0, 256]

        hist = cv2.calcHist(
            [hsv],
            [channel],
            None,
            [32],
            hist_range
        )

        hist = cv2.normalize(
            hist,
            hist
        ).flatten()

        hist_features.extend(hist)

    hist_features = np.array(hist_features)


    # ========================================================
    # 2. STATISTICAL FEATURES
    # 15 FEATURES
    # ========================================================

    stats = []

    for channel in range(3):

        channel_data = hsv[:, :, channel]

        stats.append(np.mean(channel_data))
        stats.append(np.std(channel_data))
        stats.append(np.min(channel_data))
        stats.append(np.max(channel_data))
        stats.append(np.median(channel_data))

    stats = np.array(stats)


    # ========================================================
    # 3. EDGE FEATURES
    # 3 FEATURES
    # ========================================================

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    edges = cv2.Canny(
        gray,
        100,
        200
    )

    edge_features = np.array([
        np.mean(edges > 0),
        np.std(edges),
        np.sum(edges > 0)
    ])


    # ========================================================
    # 4. HUE MOMENTS
    # 4 FEATURES
    # ========================================================

    hue = hsv[:, :, 0].astype(np.float32)

    hue_mean = np.mean(hue)
    hue_std = np.std(hue)

    hue_centered = hue - hue_mean

    hue_skewness = np.mean(
        hue_centered ** 3
    ) / (hue_std ** 3 + 1e-8)

    hue_kurtosis = np.mean(
        hue_centered ** 4
    ) / (hue_std ** 4 + 1e-8)

    hue_features = np.array([
        hue_mean,
        hue_std,
        hue_skewness,
        hue_kurtosis
    ])


    # ========================================================
    # 5. HOG FEATURES
    # ========================================================

    hog_features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys"
    )


    # ========================================================
    # 6. COMBINE ALL FEATURES
    # ========================================================

    features = np.concatenate([
        hist_features,
        stats,
        edge_features,
        hue_features,
        hog_features
    ])

    return features


# ============================================================
# FIND CAT AND DOG FILES
# ============================================================

files = sorted(
    os.listdir(DATA_FOLDER)
)

cat_files = [
    f for f in files
    if f.lower().startswith("cat")
    and f.lower().endswith(
        (".jpg", ".jpeg", ".png")
    )
]

dog_files = [
    f for f in files
    if f.lower().startswith("dog")
    and f.lower().endswith(
        (".jpg", ".jpeg", ".png")
    )
]


# ============================================================
# SELECT DATA
# ============================================================

cat_files = cat_files[:IMAGES_PER_CLASS]
dog_files = dog_files[:IMAGES_PER_CLASS]

selected_files = cat_files + dog_files

print("Selected cats:", len(cat_files))
print("Selected dogs:", len(dog_files))


# ============================================================
# EXTRACT FEATURES
# ============================================================

X = []
y = []

print("\nExtracting features...")
print("----------------------")

for i, filename in enumerate(selected_files):

    path = os.path.join(
        DATA_FOLDER,
        filename
    )

    features = extract_features(path)

    if features is None:
        continue

    X.append(features)

    if filename.lower().startswith("cat"):
        y.append(0)

    else:
        y.append(1)

    if (i + 1) % 500 == 0:
        print(
            "Processed:",
            i + 1,
            "/",
            len(selected_files)
        )


X = np.array(X)
y = np.array(y)


# ============================================================
# DATASET INFORMATION
# ============================================================

print("\nDataset information")
print("-------------------")

print("Number of images:", len(X))
print("Original feature vector size:", X.shape[1])
print("Cats:", np.sum(y == 0))
print("Dogs:", np.sum(y == 1))


# ============================================================
# FEATURE VECTOR INFORMATION
# ============================================================

print("\nFeature vector information")
print("--------------------------")

print("Number of features:", X.shape[1])
print("First 20 features:", X[0][:20])


# ============================================================
# STRATIFIED 5-FOLD CROSS VALIDATION
# ============================================================

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=42
)


# ============================================================
# STORE RESULTS
# ============================================================

fold_accuracies = []

all_true = []
all_pred = []

pca_features_per_fold = []


# ============================================================
# CROSS VALIDATION
# ============================================================

for fold, (train_index, test_index) in enumerate(
    skf.split(X, y),
    start=1
):

    print("\n================================")
    print("Fold", fold, "/", N_SPLITS)
    print("================================")


    # --------------------------------------------------------
    # TRAIN / TEST DATA
    # --------------------------------------------------------

    X_train = X[train_index]
    X_test = X[test_index]

    y_train = y[train_index]
    y_test = y[test_index]


    print("Training images:", len(X_train))
    print("Testing images:", len(X_test))


    # ========================================================
    # STANDARD SCALER
    # ========================================================

    scaler = StandardScaler()

    X_train = scaler.fit_transform(
        X_train
    )

    X_test = scaler.transform(
        X_test
    )


    # ========================================================
    # PCA
    # ========================================================

    print("\nApplying PCA...")

    pca = PCA(
        n_components=PCA_VARIANCE,
        random_state=42
    )

    X_train_pca = pca.fit_transform(
        X_train
    )

    X_test_pca = pca.transform(
        X_test
    )


    # --------------------------------------------------------
    # PCA INFORMATION
    # --------------------------------------------------------

    n_features = X_train_pca.shape[1]

    explained_variance = (
        np.sum(
            pca.explained_variance_ratio_
        )
    )

    pca_features_per_fold.append(
        n_features
    )


    print(
        "Original features:",
        X_train.shape[1]
    )

    print(
        "PCA features:",
        n_features
    )

    print(
        "Explained variance:",
        explained_variance
    )


    # ========================================================
    # SVM
    # ========================================================

    print("\nTraining SVM...")

    svm = SVC(
        kernel="rbf",
        C=10,
        gamma="scale"
    )

    svm.fit(
        X_train_pca,
        y_train
    )


    # ========================================================
    # PREDICTION
    # ========================================================

    y_pred = svm.predict(
        X_test_pca
    )


    # ========================================================
    # FOLD ACCURACY
    # ========================================================

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    fold_accuracies.append(
        accuracy
    )

    print("\nFold Accuracy")
    print("-------------")

    print(accuracy)


    # ========================================================
    # STORE PREDICTIONS
    # ========================================================

    all_true.extend(
        y_test
    )

    all_pred.extend(
        y_pred
    )


# ============================================================
# CONVERT RESULTS
# ============================================================

all_true = np.array(
    all_true
)

all_pred = np.array(
    all_pred
)


# ============================================================
# FINAL RESULTS
# ============================================================

print("\n\n================================")
print("PCA + SVM CROSS-VALIDATION")
print("================================")


# ============================================================
# FOLD ACCURACIES
# ============================================================

print("\nFold Accuracies")
print("---------------")

for i, accuracy in enumerate(
    fold_accuracies,
    start=1
):

    print(
        "Fold",
        i,
        ":",
        accuracy
    )


# ============================================================
# MEAN ACCURACY
# ============================================================

mean_accuracy = np.mean(
    fold_accuracies
)

std_accuracy = np.std(
    fold_accuracies
)


print("\nMean Accuracy")
print("-------------")

print(mean_accuracy)


print("\nStandard Deviation")
print("------------------")

print(std_accuracy)


# ============================================================
# PCA INFORMATION
# ============================================================

print("\nPCA Features Per Fold")
print("---------------------")

for i, n_features in enumerate(
    pca_features_per_fold,
    start=1
):

    print(
        "Fold",
        i,
        ":",
        n_features
    )


print("\nAverage PCA Features")
print("--------------------")

print(
    np.mean(
        pca_features_per_fold
    )
)


# ============================================================
# OVERALL CONFUSION MATRIX
# ============================================================

print("\nConfusion Matrix")
print("----------------")

print(
    confusion_matrix(
        all_true,
        all_pred,
        labels=[0, 1]
    )
)


# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print("\nClassification Report")
print("---------------------")

print(
    classification_report(
        all_true,
        all_pred,
        labels=[0, 1],
        target_names=[
            "Cat",
            "Dog"
        ]
    )
)

Selected cats: 4000
Selected dogs: 4000

Extracting features...
----------------------
Processed: 500 / 8000
Processed: 1000 / 8000
Processed: 1500 / 8000
Processed: 2000 / 8000
Processed: 2500 / 8000
Processed: 3000 / 8000
Processed: 3500 / 8000
Processed: 4000 / 8000
Processed: 4500 / 8000
Processed: 5000 / 8000
Processed: 5500 / 8000
Processed: 6000 / 8000
Processed: 6500 / 8000
Processed: 7000 / 8000
Processed: 7500 / 8000
Processed: 8000 / 8000

Dataset information
-------------------
Number of images: 8000
Original feature vector size: 1882
Cats: 4000
Dogs: 4000

Feature vector information
--------------------------
Number of features: 1882
First 20 features: [3.12163174e-04 3.74595821e-03 1.18622005e-01 9.81441021e-01
 1.49838328e-01 1.15500372e-02 9.98922158e-03 1.56081584e-03
 9.36489552e-04 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 3.12163174e-04 0.00000000e+00 0.00000000e+00
 0.00000000e+00 3.12163174e-04 0.00000000e+00 0.00000000e+00]

Fold 1 / 5
Training

In [2]:
import cv2
import numpy as np
import os

from skimage.feature import hog

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)


# ============================================================
# SETTINGS
# ============================================================

DATA_FOLDER = r"C:\Users\PC-LAB1\Desktop\shok\pictures\train"

IMAGES_PER_CLASS = 4000

N_SPLITS = 5


# ============================================================
# FEATURE EXTRACTION
# ============================================================

def extract_features(image_path):

    img = cv2.imread(image_path)

    if img is None:
        return None

    img = cv2.resize(img, (64, 64))

    # ========================================================
    # 1. HSV HISTOGRAM
    # 96 FEATURES
    # ========================================================

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    hist_features = []

    for channel in range(3):

        if channel == 0:
            hist_range = [0, 180]
        else:
            hist_range = [0, 256]

        hist = cv2.calcHist(
            [hsv],
            [channel],
            None,
            [32],
            hist_range
        )

        hist = cv2.normalize(
            hist,
            hist
        ).flatten()

        hist_features.extend(hist)

    hist_features = np.array(hist_features)

    # ========================================================
    # 2. STATISTICAL FEATURES
    # 15 FEATURES
    # ========================================================

    stats = []

    for channel in range(3):

        channel_data = hsv[:, :, channel]

        stats.append(np.mean(channel_data))
        stats.append(np.std(channel_data))
        stats.append(np.min(channel_data))
        stats.append(np.max(channel_data))
        stats.append(np.median(channel_data))

    stats = np.array(stats)

    # ========================================================
    # 3. EDGE FEATURES
    # 3 FEATURES
    # ========================================================

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    edges = cv2.Canny(
        gray,
        100,
        200
    )

    edge_features = np.array([
        np.mean(edges > 0),
        np.std(edges),
        np.sum(edges > 0)
    ])

    # ========================================================
    # 4. HUE MOMENTS
    # 4 FEATURES
    # ========================================================

    hue = hsv[:, :, 0].astype(np.float32)

    hue_mean = np.mean(hue)
    hue_std = np.std(hue)

    hue_centered = hue - hue_mean

    hue_skewness = np.mean(
        hue_centered ** 3
    ) / (hue_std ** 3 + 1e-8)

    hue_kurtosis = np.mean(
        hue_centered ** 4
    ) / (hue_std ** 4 + 1e-8)

    hue_features = np.array([
        hue_mean,
        hue_std,
        hue_skewness,
        hue_kurtosis
    ])

    # ========================================================
    # 5. HOG FEATURES
    # 1764 FEATURES
    # ========================================================

    hog_features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys"
    )

    # ========================================================
    # 6. COMBINE
    # ========================================================

    features = np.concatenate([
        hist_features,
        stats,
        edge_features,
        hue_features,
        hog_features
    ])

    return features


# ============================================================
# FIND FILES
# ============================================================

files = sorted(
    os.listdir(DATA_FOLDER)
)

cat_files = [
    f for f in files
    if f.lower().startswith("cat")
    and f.lower().endswith(
        (".jpg", ".jpeg", ".png")
    )
]

dog_files = [
    f for f in files
    if f.lower().startswith("dog")
    and f.lower().endswith(
        (".jpg", ".jpeg", ".png")
    )
]


# ============================================================
# SELECT 4000 CATS + 4000 DOGS
# ============================================================

cat_files = cat_files[:IMAGES_PER_CLASS]
dog_files = dog_files[:IMAGES_PER_CLASS]

selected_files = cat_files + dog_files

print("Selected cats:", len(cat_files))
print("Selected dogs:", len(dog_files))


# ============================================================
# EXTRACT FEATURES
# ============================================================

X = []
y = []

print("\nExtracting features...")
print("----------------------")

for i, filename in enumerate(selected_files):

    path = os.path.join(
        DATA_FOLDER,
        filename
    )

    features = extract_features(path)

    if features is None:
        continue

    X.append(features)

    if filename.lower().startswith("cat"):
        y.append(0)
    else:
        y.append(1)

    if (i + 1) % 500 == 0:
        print(
            "Processed:",
            i + 1,
            "/",
            len(selected_files)
        )


X = np.array(X)
y = np.array(y)


# ============================================================
# DATASET INFORMATION
# ============================================================

print("\nDataset information")
print("-------------------")

print("Number of images:", len(X))
print("Feature vector size:", X.shape[1])
print("Cats:", np.sum(y == 0))
print("Dogs:", np.sum(y == 1))


# ============================================================
# FEATURE INFORMATION
# ============================================================

print("\nFeature vector information")
print("--------------------------")

print("Number of features:", X.shape[1])
print("First 20 features:", X[0][:20])


# ============================================================
# 5-FOLD CROSS VALIDATION
# ============================================================

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_accuracies = []

all_true = []
all_pred = []


# ============================================================
# CROSS VALIDATION
# ============================================================

for fold, (train_index, test_index) in enumerate(
    skf.split(X, y),
    start=1
):

    print("\n================================")
    print("Fold", fold, "/ 5")
    print("================================")

    X_train = X[train_index]
    X_test = X[test_index]

    y_train = y[train_index]
    y_test = y[test_index]

    print("Training images:", len(X_train))
    print("Testing images:", len(X_test))


    # ========================================================
    # SCALING
    # ========================================================

    scaler = StandardScaler()

    X_train = scaler.fit_transform(
        X_train
    )

    X_test = scaler.transform(
        X_test
    )


    # ========================================================
    # YOUR BEST SVM
    # ========================================================

    svm = SVC(
        kernel="rbf",
        C=10,
        gamma="scale"
    )


    # ========================================================
    # TRAIN
    # ========================================================

    print("Training SVM...")

    svm.fit(
        X_train,
        y_train
    )


    # ========================================================
    # PREDICT
    # ========================================================

    y_pred = svm.predict(
        X_test
    )


    # ========================================================
    # ACCURACY
    # ========================================================

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    fold_accuracies.append(
        accuracy
    )

    all_true.extend(y_test)
    all_pred.extend(y_pred)

    print("\nFold Accuracy:")
    print(accuracy)


# ============================================================
# FINAL RESULTS
# ============================================================

all_true = np.array(all_true)
all_pred = np.array(all_pred)

mean_accuracy = np.mean(
    fold_accuracies
)

std_accuracy = np.std(
    fold_accuracies
)


print("\n\n================================")
print("5-FOLD CROSS-VALIDATION RESULTS")
print("================================")


print("\nFold Accuracies")
print("----------------")

for i, accuracy in enumerate(
    fold_accuracies,
    start=1
):

    print(
        f"Fold {i}: {accuracy:.6f}"
    )


print("\nMean Accuracy")
print("-------------")

print(
    f"{mean_accuracy:.6f}"
)


print("\nStandard Deviation")
print("------------------")

print(
    f"{std_accuracy:.6f}"
)


print("\nMean Accuracy ± Standard Deviation")
print("-----------------------------------")

print(
    f"{mean_accuracy:.4f} ± {std_accuracy:.4f}"
)


# ============================================================
# CONFUSION MATRIX
# ============================================================

print("\nConfusion Matrix")
print("----------------")

print(
    confusion_matrix(
        all_true,
        all_pred,
        labels=[0, 1]
    )
)


# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print("\nClassification Report")
print("---------------------")

print(
    classification_report(
        all_true,
        all_pred,
        labels=[0, 1],
        target_names=[
            "Cat",
            "Dog"
        ]
    )
)

Selected cats: 4000
Selected dogs: 4000

Extracting features...
----------------------
Processed: 500 / 8000
Processed: 1000 / 8000
Processed: 1500 / 8000
Processed: 2000 / 8000
Processed: 2500 / 8000
Processed: 3000 / 8000
Processed: 3500 / 8000
Processed: 4000 / 8000
Processed: 4500 / 8000
Processed: 5000 / 8000
Processed: 5500 / 8000
Processed: 6000 / 8000
Processed: 6500 / 8000
Processed: 7000 / 8000
Processed: 7500 / 8000
Processed: 8000 / 8000

Dataset information
-------------------
Number of images: 8000
Feature vector size: 1882
Cats: 4000
Dogs: 4000

Feature vector information
--------------------------
Number of features: 1882
First 20 features: [3.12163174e-04 3.74595821e-03 1.18622005e-01 9.81441021e-01
 1.49838328e-01 1.15500372e-02 9.98922158e-03 1.56081584e-03
 9.36489552e-04 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 3.12163174e-04 0.00000000e+00 0.00000000e+00
 0.00000000e+00 3.12163174e-04 0.00000000e+00 0.00000000e+00]

Fold 1 / 5
Training images: 